# មេរៀន ០៣ - លំនាំរចនាប័ទ្មភ្នាក់ងារផ្ទាល់ខ្លួន

នៅក្នុងមេរៀននេះ យើងនឹងស្វែងយល់ពីលំនាំរចនាប័ទ្មមូលដ្ឋានបីសម្រាប់កសាងភ្នាក់ងារតំបន់ប្រសិទ្ធភាព:

1. **ជំនួយការណ៍ភ្នាក់ងារដែលច្បាស់លាស់** — ការបង្កើតសំនួរជាក់លាក់ដែលកំណត់តួនាទី ដើម្បីណែនាំអាកប្បកិរិយារបស់ភ្នាក់ងារ
2. **លទ្ធផលរចនាសម្ព័ន្ធជាមួយគំរូ Pydantic** — ធានាថាភ្នាក់ងារផ្តល់លទ្ធផលដែលអាចទាយទ្រង់ទ្រាយ និងបានបញ្ជាក់
3. **ភ្នាក់ងារត្រូវមានតួនាទីតែមួយ** — រចនាភ្នាក់ងារតែមួយដែលធ្វើរឿងមួយយ៉ាងល្អ

យើងនឹងអនុវត្តលំនាំនីមួយៗទៅលើសេណារីយោ **អ្នកផ្តល់អនុសាសន៍ចំនួនកន្លែងធ្វើដំណើរ** ដែលបន្ថែមឡើងៗក្នុងការបង្កើតប្រព័ន្ធដែលអាចផ្តល់អនុសាសន៍កន្លែង ស្វែងរកភាពអាចប្រើបាន និងគ្រប់គ្រងការរៀបចំដំណើរ។


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## គំរូទី 1៖ សេចក្តីណែនាំច្បាស់លាស់សម្រាប់ភ្នាក់ងារ

គំរូដែលមានឥទ្ធិពលខ្លាំងបំផុត ក៏ជាគំរូសាមញ្ញផងដែរ៖ ការសរសេរសេចក្តីណែនាំច្បាស់លាស់ និងលម្អិតសម្រាប់ភ្នាក់ងាររបស់អ្នក។

សេចក្ដីណែនាំល្អកំណត់៖
- **នរណា** ដែលជាភ្នាក់ងារ (បុគ្គលិកលក្ខណៈ និងសម្លេង)
- **អ្វី** ដែលវាគួរធ្វើ (ភារកិច្ចជាគ្រប់ជំហាន)
- **តើវាគួរប្រព្រឹត្តិប្រាស** បានយ៉ាងដូចម្តេច (បញ្ហា និងរចនាបថ)

ខាងក្រោម យើងបង្កើតភ្នាក់ងារតំណាងដំណើរកំសាន្តដែលមានសេចក្តីណែនាំច្បាស់លាស់ដែលកំណត់រាល់ចម្លើយដែលវាបង្កើតឡើង។


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## លំនាំទី 2: ការចេញផ្សាយរាងរចនាសម្ព័ន្ធជាមួយគំរូ Pydantic

អត្ថបទរាងសេរីមានប្រយោជន៍សម្រាប់ការសន្ទនា ប៉ុន្តែប្រព័ន្ធក្រោយសាម៊តត្រូវការទិន្នន័យមានរចនាសម្ព័ន្ធ។
ដោយបង្រួបបង្រួម **គំរូ Pydantic** ជាមួយ **មុខងារ​ឧបករណ៍** យើងអាច:

- កំណត់រចនាសម្ព័ន្ធច្បាស់លាស់សម្រាប់លទ្ធផលរបស់ភ្នាក់ងារ
- បញ្ជាក់ទទួលស្វ័យប្រវត្តិ
- បញ្ចូលលទ្ធផលភ្នាក់ងារចូលក្នុងตรรกะកម្មវិធីយ៉ាងទៀងទាត់

គន្លឹះក្នុងការអនុវត្តគឺបញ្ជូន `response_format` ពេលយើងបើកប្រើភ្នាក់ងារ។ វារឹតបណ្តឹងឱ្យ
ម៉ូឌែលត្រូវតែបញ្ជូនវត្ថុ `TravelRecommendations` ដែលបានបញ្ជាក់ទោល (អាចប្រើបាននៅលើ `response.value`)
ជំនួសអត្ថបទរាងសេរី។ ឧបករណ៍ `get_destination_details` ក៏ចេញវត្ថុឆ្ពោះទៅកាន់
`DestinationRecommendation` ដែលបានចែងប្រភេទ ហើយដូច្នេះទិន្នន័យនៅតែមានរចនាសម្ព័ន្ធពីចុងដើមដល់ចុងក្រោយ។


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## ការគំរូទី ៣៖ អ្នកតំណាងមានភារកិច្ចតែមួយ

ការងារលំបាកមានអត្ថប្រយោជន៍ពីការបំបែកការងារចែកចាយទៅឲ្យអ្នកតំណាងមួយចំនួនដែលមានភារកិច្ចតែមួយ៖

- អ្នកជំនាញ **គោលដៅ** ដែលស្គាល់អំពីកន្លែង និងការចូលប្រើ
- អ្នករៀបចំ **ក្រោមការដឹកជញ្ជូន** ដែលដឹកនាំការហោះហើរ សណ្ឋាគារ និងកាលវិភាគ

នេះដូចនឹងគោលការណ៍វិស្វកម្មកម្មវិធី *ការបំបែកវិស័យចំណាប់អារម្មណ៍* — អ្នកតំណាងនីមួយៗងាយស្រួលសាកល្បង ថែទាំ និងបង្កេីតឱ្យមានកែលម្អដោយសេរី។


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## សង្ខេប

ក្នុងមេរៀននេះ យើងបានអនុវត្តលំនាំការរចនាតំណាងភ្នាក់ងារបីទៅលើស្នាដៃអ្នកផ្តល់អនុសាសន៍ធ្វើដំណើរ៖

| លំនាំ | គំនិតសំខាន់ | អត្ថប្រយោជន៍ |
|---|---|---|
| **ការណែនាំច្បាស់លាស់** | កំណត់បុគ្គលិកលក្ខណៈ, ភារកិច្ច និងការរឹងបន្តឹងនៅលើ | ការប្រព្រឹត្តលក្ខណៈភ្នាក់ងារតាមម៉ាកយ៉ាងរឹងរូស |
| **ចេញ​ផល​គ្រប់​គ្រាន់** | ប្រើម៉ូដែល Pydantic ក្នុងទ្រង់ទ្រាយចម្លើយ | លទ្ធផលបានផ្ទៀងផ្ទាត់និងអាចអានដោយម៉ាស៊ីន |
| **ភារកិច្ចតែមួយ** | ផ្តល់ការងារតែមួយជាការផ្តោតសំខាន់សម្រាប់ភ្នាក់ងារ | ងាយស្រួលសាកល្បង, ទប់ស្កាត់ និងរួមបញ្ចូល |

លំនាំទាំងនេះរួមបញ្ចូលគ្នាយ៉ាងធម្មជាតិ - អ្នកអាចផ្សំការណែនាំច្បាស់លាស់ជាមួយការចេញ​ផល​គ្រប់​គ្រាន់នៅក្នុងភ្នាក់ងារមានភារកិច្ចតែមួយដើម្បីបង្កើតប្រព័ន្ធរឹងមាំ ដែលរួចរាល់សម្រាប់ផលិតកម្ម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
